# 03 — Train the PyTorch and TensorFlow neural baselines

This notebook uses the current trainer APIs in the package. Both frameworks receive the same model-fitting, early-stopping validation, calibration and held-out test rows. The scaler is fitted only on the model-fitting rows. The calibration rows are not used in training or early stopping.

This is a small educational benchmark, not a clinical model.

## Install the optional dependencies

From the repository root, install the project and neural extras:

```bash
pip install -e ".[neural]"
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from diabetes_ml.data import create_experiment_splits
from diabetes_ml.evaluation.metrics import calculate_regression_metrics
from diabetes_ml.neural.neural_data import prepare_neural_datasets
from diabetes_ml.neural.pytorch_model import PyTorchTrainer
from diabetes_ml.neural.tensorflow_model import TensorFlowTrainer

splits = create_experiment_splits(test_size=0.20, random_state=42)
data = prepare_neural_datasets(
    splits.X_train,
    splits.y_train,
    splits.X_test,
    splits.y_test,
    val_fraction=0.20,
    cal_fraction=0.20,
    random_state=42,
)

print(f"Model-fitting rows: {len(data.X_train)}")
print(f"Early-stopping validation rows: {len(data.X_val)}")
print(f"Conformal calibration rows (not used below): {len(data.X_cal)}")
print(f"Held-out test rows: {len(data.X_test)}")

## Train the PyTorch model

The trainer restores the weights from the epoch with the lowest validation MSE. The test set is used only after fitting is complete.

In [ ]:
pytorch = PyTorchTrainer(
    in_features=data.X_train.shape[1],
    epochs=200,
    patience=20,
    batch_size=32,
    random_state=42,
)
pytorch.fit(data.X_train, data.y_train, data.X_val, data.y_val)
pytorch_predictions = pytorch.predict(data.X_test)
pytorch_metrics = calculate_regression_metrics(data.y_test.ravel(), pytorch_predictions)
print("PyTorch metrics:", pytorch_metrics)
print("PyTorch best validation epoch:", min(pytorch.history, key=lambda row: row["val_mse"])["epoch"])

## Train the TensorFlow/Keras model

The TensorFlow trainer also restores its best validation weights. It uses exactly the same partitions as the PyTorch model.

In [ ]:
tensorflow_model = TensorFlowTrainer(
    in_features=data.X_train.shape[1],
    epochs=200,
    patience=20,
    batch_size=32,
    random_state=42,
)
tensorflow_model.fit(data.X_train, data.y_train, data.X_val, data.y_val)
tensorflow_predictions = tensorflow_model.predict(data.X_test)
tensorflow_metrics = calculate_regression_metrics(data.y_test.ravel(), tensorflow_predictions)
print("TensorFlow metrics:", tensorflow_metrics)
print("TensorFlow best validation epoch:", min(tensorflow_model.history_records, key=lambda row: row["val_mse"])["epoch"])

In [ ]:
comparison = pd.DataFrame([
    {"framework": "PyTorch", **pytorch_metrics},
    {"framework": "TensorFlow/Keras", **tensorflow_metrics},
])
display(comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, history, label in [
    (axes[0], pytorch.history, "PyTorch"),
    (axes[1], tensorflow_model.history_records, "TensorFlow/Keras"),
]:
    axis.plot([row["epoch"] for row in history], [row["train_mse"] for row in history], label="Training MSE")
    axis.plot([row["epoch"] for row in history], [row["val_mse"] for row in history], label="Validation MSE")
    axis.set_title(label)
    axis.set_xlabel("Epoch")
    axis.set_ylabel("MSE")
    axis.legend()
fig.tight_layout()
plt.show()

## Interpretation and limits

- Compare training and validation curves for signs of underfitting or overfitting.
- MAE and RMSE are expressed in the target's units; R² may be negative on held-out data.
- The results depend on one split, one small MLP configuration and the selected training controls. They are not evidence that one framework is generally better.
- This notebook reports point-prediction metrics only. It does not fit or evaluate conformal intervals; the reserved calibration subset is intentionally left unused here.
- Do not interpret this benchmark as clinical validation or as a model for individual patient care.